# Transformações e Escalonamento - Parte 2
## Técnicas Avançadas de Transformação

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 07 - Parte 2: Técnicas Avançadas

---

> **📝 Nota**: Certifique-se de ter completado a Parte 1 antes de prosseguir com estas técnicas avançadas.

---

# ==================================================
# INTRODUÇÃO ÀS TÉCNICAS AVANÇADAS
# ==================================================

OBJETIVOS DESTA PARTE:
- Explorar métodos avançados de escalonamento e transformação
- Compreender o impacto do escalonamento em PCA
- Conhecer as melhores práticas e recomendações

In [ ]:
# @title
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    RobustScaler,
    Normalizer,
    MaxAbsScaler,
    PowerTransformer,
    QuantileTransformer
)
from sklearn.datasets import fetch_california_housing
from sklearn.decomposition import PCA
from IPython.display import Markdown

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

In [ ]:
# @title
# Carregando o dataset para uso nesta parte
california = fetch_california_housing()
X = pd.DataFrame(california.data, columns=california.feature_names)
y = california.target

print("Dataset carregado com sucesso!")
print(f"Shape: {X.shape}")
print(f"Features: {list(X.columns)}")

# ==================================================
# 1. OUTROS MÉTODOS DE ESCALONAMENTO
# ==================================================

Além das técnicas fundamentais (Min-Max, Standardization e Robust Scaling) vistas na Parte 1, existem outros métodos úteis em situações específicas:

### 📊 Comparação Rápida dos Métodos Avançados

| Método | Tipo | Transforma | Melhor Para | Preserva |
|--------|------|------------|-------------|----------|
| **MaxAbs Scaler** | Escalonamento | Linear | Dados esparsos | Zeros e esparsidade |
| **Normalizer** | Escalonamento | Linear (por amostra) | Similaridade direcional | Direção, não magnitude |
| **Transformações Simples** | Matemática | Não-linear | Reduzir assimetria | Interpretabilidade |
| **Power Transformer** | Matemática | Não-linear otimizada | Normalizar distribuições | Relações monotônicas |
| **Quantile Transformer** | Estatística | Não-linear | Outliers extremos | Ordem (ranks) |

### Resumo de cada método:

#### 1.1 MaxAbs Scaler
- Escala pelo valor absoluto máximo
- Preserva zeros e esparsidade
- Útil para dados esparsos

#### 1.2 Normalizer
- Normaliza cada amostra (linha) independentemente
- Diferentes normas: L1, L2, Max
- Útil quando a magnitude não importa, apenas a direção

#### 1.3 Transformações Matemáticas Simples
- Log, Sqrt, Recíproca
- Interpretáveis e reversíveis
- Eficazes para reduzir assimetria

#### 1.4 Power Transformer
- Box-Cox e Yeo-Johnson
- Torna dados mais "normais"
- Encontra automaticamente melhor transformação

#### 1.5 Quantile Transformer
- Força distribuição uniforme ou normal
- Extremamente robusto a outliers
- Pode distorcer relações entre variáveis

## 1.1 MaxAbs Scaler

### 📚 Conceito e Teoria

**MaxAbs Scaler** escala cada feature dividindo pelo valor absoluto máximo dessa feature. A transformação é dada por:

$$X_{maxabs} = \frac{X}{|X_{max}|}$$

Onde:
- $X$ é o valor original
- $|X_{max}|$ é o valor absoluto máximo da feature
- $X_{maxabs}$ é o valor escalado (estará no intervalo [-1, 1])

### 🎯 Características Principais

- **Preserva zeros**: Valores zero permanecem zero após transformação
- **Preserva esparsidade**: Ideal para dados esparsos
- **Preserva sinal**: Mantém valores positivos e negativos
- **Intervalo resultante**: [-1, 1]

### 💡 Quando Usar

- **Dados esparsos**: Especialmente útil para matrizes esparsas (ex: TF-IDF, bag-of-words)
- **Preservar zeros**: Quando zeros têm significado especial
- **Dados já centrados**: Quando os dados já estão centrados em zero
- **Alternativa ao StandardScaler**: Para dados esparsos onde StandardScaler destruiria a esparsidade

In [ ]:
# @title
# Implementando MaxAbs Scaler
max_abs_scaler = MaxAbsScaler()
X_maxabs = max_abs_scaler.fit_transform(X)
X_maxabs_df = pd.DataFrame(X_maxabs, columns=X.columns)

print("Estatísticas após MaxAbs Scaling:")
print(X_maxabs_df.describe())

In [ ]:
# @title
# Verificando valores máximos absolutos (devem estar próximos de 1 ou -1)
print("Valores máximos absolutos:")
for col in X_maxabs_df.columns:
    abs_max = max(abs(X_maxabs_df[col].min()), abs(X_maxabs_df[col].max()))
    print(f"{col}: {abs_max:.4f}")

## 1.2 Normalizer (Normalização L1, L2, Max)

### 📚 Conceito e Teoria

**Normalizer** é diferente dos outros scalers — ele normaliza **amostras** (linhas) individualmente, não features (colunas). Cada amostra é escalada para ter **norma unitária**, preservando a **direção** do vetor, mas ajustando a **magnitude**.

- **Norma L1 (Manhattan):**  
$$
X_{L1} = \frac{X}{||X||_1} = \frac{X}{\sum_{i}|x_i|}
$$

- **Norma L2 (Euclidiana):**  
$$
X_{L2} = \frac{X}{||X||_2} = \frac{X}{\sqrt{\sum_{i}x_i^2}}
$$

- **Norma Max (Infinity Norm):**  
$$
X_{max} = \frac{X}{||X||_\infty} = \frac{X}{\max_{i} |x_i|}
$$

---

### 🎯 Características Principais

- **Normalização por amostra:** Cada linha é normalizada independentemente.  
- **Direção vs magnitude:** Preserva a direção do vetor, normaliza a magnitude.  
- **Vetores unitários:** Transforma dados em vetores de norma 1.  
- **Não afeta distribuição entre amostras:** Apenas escala cada amostra individualmente.

---

### 🔹 Exemplo Numérico

Suponha uma amostra \(X = [3, 4]\):

- **L1:** $$ (\sum |x_i| = 3 + 4 = 7), normalizado: ([3/7, 4/7] \approx [0.43, 0.57] $$
- **L2:** $$ (\sqrt{3^2 + 4^2} = 5), normalizado: ([3/5, 4/5] = [0.6, 0.8])$$
- **Max:** $$(\max(|3|, |4|) = 4), normalizado: ([3/4, 4/4] = [0.75, 1])$$


> Note que **cada norma gera um único número** que é usado para dividir todos os atributos da linha.

---

### 💡 Quando Usar

- **Similaridade de cosseno:** Quando a magnitude não importa, apenas a direção.  
- **Análise de texto:** Normalização de vetores TF-IDF.  
- **Clustering direcional:** Agrupamento por direção, não magnitude.  
- **Dados de alta dimensão:** Reduz efeito do *curse of dimensionality*.  
- **Comparação de perfis:** Para comparar "formas" independente da escala.

---

### 🔹 Visualização Conceitual

Imagine cada linha como um vetor no espaço multidimensional:

- **L2:** Vetor encaixado dentro de uma **esfera unitária**.  
- **L1:** Vetor encaixado dentro de um **losango (taxi-cab)**.  
- **Max:** Vetor escalado pelo **maior valor do vetor**.  

A **direção do vetor não muda**, apenas a magnitude é ajustada.


In [ ]:
# @title
# Implementando Normalizer com diferentes normas
# L1 (soma dos valores absolutos = 1)
normalizer_l1 = Normalizer(norm='l1')
X_norm_l1 = normalizer_l1.fit_transform(X)
X_norm_l1_df = pd.DataFrame(X_norm_l1, columns=X.columns)

# L2 (raiz quadrada da soma dos quadrados = 1)
normalizer_l2 = Normalizer(norm='l2')
X_norm_l2 = normalizer_l2.fit_transform(X)
X_norm_l2_df = pd.DataFrame(X_norm_l2, columns=X.columns)

# Verificando as normas das primeiras amostras
print("Soma dos valores absolutos (norma L1) para as primeiras amostras:")
print(np.sum(np.abs(X_norm_l1[:5]), axis=1))

print("\nNorma L2 para as primeiras amostras:")
print(np.sqrt(np.sum(X_norm_l2[:5]**2, axis=1)))

In [ ]:
# @title
# Visualizando o efeito de Normalizer - ele normaliza cada amostra, não features
# Comparando Min-Max Scaling (que normaliza features) com Normalizer

# Criando Min-Max para comparação
X_minmax = MinMaxScaler().fit_transform(X)
X_minmax_df = pd.DataFrame(X_minmax, columns=X.columns)

plt.figure(figsize=(15, 5))

# Vamos usar 2 variáveis para visualização
feature1 = 'MedInc'
feature2 = 'HouseAge'

plt.subplot(1, 3, 1)
plt.scatter(X[feature1], X[feature2], alpha=0.3)
plt.title('Original')
plt.xlabel(feature1)
plt.ylabel(feature2)

plt.subplot(1, 3, 2)
plt.scatter(X_minmax_df[feature1], X_minmax_df[feature2], alpha=0.3)
plt.title('Min-Max Scaling (normaliza features)')
plt.xlabel(feature1)
plt.ylabel(feature2)

plt.subplot(1, 3, 3)
plt.scatter(X_norm_l2_df[feature1], X_norm_l2_df[feature2], alpha=0.3)
plt.title('Normalizer L2 (normaliza amostras)')
plt.xlabel(feature1)
plt.ylabel(feature2)

plt.tight_layout()
plt.show()

## 1.3 Transformações Matemáticas Simples

### 📚 Conceito e Teoria

Antes de usar métodos sofisticados, transformações matemáticas simples podem ser surpreendentemente eficazes para normalizar distribuições:

**Transformação Logarítmica:**
$$X_{log} = \log(X) \text{ ou } \log(X+1)$$
- Reduz assimetria positiva forte
- Comprime valores grandes, expande valores pequenos
- Interpretável: diferenças no log representam mudanças multiplicativas

**Transformação Raiz Quadrada:**
$$X_{sqrt} = \sqrt{X}$$
- Reduz assimetria positiva moderada
- Menos drástica que logaritmo
- Preserva zeros (√0 = 0)

**Transformação Recíproca:**
$$X_{rec} = \frac{1}{X} \text{ ou } \frac{1}{X+c}$$
- Inverte a ordem e direção da assimetria
- Transforma assimetria positiva em negativa (e vice-versa)
- Cuidado: não definida para zero (adicionar constante c)

### 🎯 Características e Efeitos

| Transformação | Assimetria Original | Efeito | Força |
|--------------|-------------------|---------|--------|
| Log | Positiva forte | Reduz drasticamente | Forte |
| Sqrt | Positiva moderada | Reduz moderadamente | Moderada |
| Recíproca | Positiva | Inverte para negativa | Muito forte |
| Recíproca | Negativa | Inverte para positiva | Muito forte |

### 💡 Quando Usar

- **Logarítmica**:
  - Dados de crescimento exponencial (população, vendas)
  - Valores monetários (salários, preços)
  - Concentrações químicas
  
- **Raiz Quadrada**:
  - Dados de contagem (número de eventos)
  - Áreas ou volumes
  - Quando log é muito forte
  
- **Recíproca**:
  - Tempos de espera ou duração
  - Velocidades ou taxas
  - Quando precisa inverter assimetria

### ⚠️ Cuidados Importantes

1. **Valores zero ou negativos**:
   - Log e sqrt precisam de valores positivos
   - Use log(x+1) ou sqrt(x+c) se necessário
   
2. **Interpretabilidade**:
   - Log: mudanças multiplicativas
   - Sqrt: relações quadráticas
   - Recíproca: inversão de escala

3. **Reversibilidade**:
   - Log → exponencial: $X = e^{X_{log}}$
   - Sqrt → quadrado: $X = X_{sqrt}^2$
   - Recíproca → recíproca: $X = 1/X_{rec}$

In [ ]:
# @title
# Demonstrando transformações matemáticas simples
# Usando a variável AveOccup que tem forte assimetria positiva

variavel_exemplo = 'AveOccup'
dados = X[variavel_exemplo].copy()

# Garantindo valores positivos para log e sqrt
dados_positivos = dados - dados.min() + 0.001 if dados.min() <= 0 else dados

# Aplicando transformações
transformacoes_simples = {
    'Original': dados,
    'Log(x+1)': np.log1p(dados_positivos),
    'Sqrt(x)': np.sqrt(dados_positivos),
    'Recíproca 1/(x+1)': 1 / (dados_positivos + 1)
}

# Visualizando o efeito
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (nome, dados_transf) in enumerate(transformacoes_simples.items()):
    axes[idx].hist(dados_transf, bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{nome}\nAssimetria: {scipy.stats.skew(dados_transf):.2f}', fontweight='bold')
    axes[idx].set_xlabel('Valor')
    axes[idx].set_ylabel('Frequência')

    # Adicionar estatísticas
    axes[idx].axvline(dados_transf.mean(), color='red', linestyle='--', label=f'Média: {dados_transf.mean():.2f}')
    axes[idx].axvline(dados_transf.median(), color='green', linestyle='--', label=f'Mediana: {dados_transf.median():.2f}')
    axes[idx].legend(fontsize=8)

plt.suptitle(f'Transformações Matemáticas Simples - {variavel_exemplo}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Análise quantitativa
print("📊 ANÁLISE DO EFEITO DAS TRANSFORMAÇÕES SIMPLES")
print("=" * 60)
print(f"{'Transformação':<20} {'Assimetria':>12} {'Efeito':<20}")
print("-" * 60)

for nome, dados_transf in transformacoes_simples.items():
    skew = scipy.stats.skew(dados_transf)
    if nome == 'Original':
        efeito = "Baseline"
    elif abs(skew) < 0.5:
        efeito = "✅ Excelente"
    elif abs(skew) < 1:
        efeito = "👍 Bom"
    elif abs(skew) < 2:
        efeito = "⚠️ Moderado"
    else:
        efeito = "❌ Insuficiente"

    print(f"{nome:<20} {skew:>12.2f} {efeito:<20}")

print("\n💡 OBSERVAÇÕES:")
print("• Log é muito eficaz para assimetria positiva forte")
print("• Sqrt é mais suave, útil para assimetria moderada")
print("• Recíproca inverte a assimetria - útil em casos específicos")

## 1.3 Power Transformer

### 📚 Conceito e Teoria

**Power Transformer** aplica transformações de potência para tornar dados mais próximos de uma distribuição normal. Duas famílias principais:

**Box-Cox** (apenas para dados positivos):
$$y_i^{(\lambda)} = \begin{cases}
\frac{y_i^{\lambda} - 1}{\lambda} & \text{se } \lambda \neq 0 \\
\ln(y_i) & \text{se } \lambda = 0
\end{cases}$$

**Yeo-Johnson** (funciona com dados positivos e negativos):
$$y_i^{(\lambda)} = \begin{cases}
\frac{(y_i + 1)^{\lambda} - 1}{\lambda} & \text{se } \lambda \neq 0, y \geq 0 \\
\ln(y_i + 1) & \text{se } \lambda = 0, y \geq 0 \\
\frac{-((-y_i + 1)^{2-\lambda} - 1)}{2 - \lambda} & \text{se } \lambda \neq 2, y < 0 \\
-\ln(-y_i + 1) & \text{se } \lambda = 2, y < 0
\end{cases}$$

O parâmetro $\lambda$ é estimado via máxima verossimilhança para cada feature.

### 🎯 Características Principais

- **Reduz assimetria**: Torna distribuições assimétricas mais simétricas
- **Estabiliza variância**: Reduz heterocedasticidade
- **Aproxima normalidade**: Útil para modelos que assumem normalidade
- **Parâmetro λ otimizado**: Encontra automaticamente a melhor transformação

### 💡 Quando Usar

- **Distribuições assimétricas**: Especialmente com cauda longa à direita
- **Pressupostos de normalidade**: Quando o modelo assume distribuição normal
- **Antes de PCA**: Para melhorar a interpretabilidade dos componentes
- **Regressão linear**: Melhora pressupostos de normalidade dos resíduos
- **Redução de outliers**: Transformação logarítmica reduz impacto de valores extremos

In [ ]:
# @title
# Implementando Power Transformer
# Yeo-Johnson funciona com dados positivos e negativos
pt_yj = PowerTransformer(method='yeo-johnson')
X_pt_yj = pt_yj.fit_transform(X)
X_pt_yj_df = pd.DataFrame(X_pt_yj, columns=X.columns)

# Box-Cox funciona apenas com dados positivos
# Vamos garantir que todos os dados sejam positivos (adicionando uma constante)
X_pos = X.copy()
for col in X_pos.columns:
    min_val = X_pos[col].min()
    if min_val <= 0:
        X_pos[col] = X_pos[col] - min_val + 0.1  # Garantindo valores positivos

pt_bc = PowerTransformer(method='box-cox')
X_pt_bc = pt_bc.fit_transform(X_pos)
X_pt_bc_df = pd.DataFrame(X_pt_bc, columns=X.columns)

print("Estatísticas após Power Transform (Yeo-Johnson):")
print(X_pt_yj_df.describe())

In [ ]:
# @title
# Verificando os parâmetros lambda encontrados para cada transformação
print("Parâmetros lambda para transformação Yeo-Johnson:")
for i, col in enumerate(X.columns):
    print(f"{col}: {pt_yj.lambdas_[i]:.4f}")

print("\nParâmetros lambda para transformação Box-Cox:")
for i, col in enumerate(X.columns):
    print(f"{col}: {pt_bc.lambdas_[i]:.4f}")

In [ ]:
# @title
# Visualizando o efeito do Power Transformer em uma variável assimétrica
skewed_feature = 'AveOccup'  # Tipicamente assimétrica

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(X[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Original')

plt.subplot(1, 3, 2)
sns.histplot(X_pt_yj_df[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Yeo-Johnson')

plt.subplot(1, 3, 3)
sns.histplot(X_pt_bc_df[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Box-Cox')

plt.tight_layout()
plt.show()

## 1.4 Quantile Transformer

### 📚 Conceito e Teoria

**Quantile Transformer** transforma features para seguir uma distribuição uniforme ou normal usando a função de distribuição acumulada (CDF-Cumulative Distribution Function). O processo:

1. **Calcula quantis**: Estima a CDF empírica dos dados
2. **Mapeia para uniforme**: Transforma valores usando seus quantis (distribuição uniforme em [0,1])
3. **Opcional - mapeia para normal**: Aplica a função quantil inversa da normal padrão

Matematicamente:
$$X_{uniform} = F_X(X)$$
$$X_{normal} = \Phi^{-1}(F_X(X))$$

Onde:
- $F_X$ é a CDF empírica de X
- $\Phi^{-1}$ é a função quantil inversa da normal padrão

### 🎯 Características Principais

- **Extremamente robusto**: Ignora completamente outliers
- **Força distribuição**: Garante distribuição uniforme ou normal
- **Não-linear**: Transformação pode distorcer relações
- **Rank-based**: Baseado apenas na ordem dos valores
- **Destrói informação de escala**: Perde informação sobre distâncias relativas

### 💡 Quando Usar

- **Outliers extremos**: Quando há muitos outliers que prejudicam outros métodos
- **Distribuições arbitrárias**: Funciona com qualquer distribuição
- **Algoritmos sensíveis**: Para algoritmos que assumem distribuição específica
- **Comparação entre features**: Quando queremos colocar todas as features na mesma escala

### ⚠️ Quando NÃO Usar

- **Relações lineares importantes**: Distorce relações lineares entre variáveis
- **Interpretabilidade**: Dificulta interpretação dos coeficientes
- **Poucos dados**: Precisa de muitos dados para estimar quantis precisamente
- **Extrapolação**: Problemático para prever valores fora do range de treino

In [ ]:
# @title
# Implementando Quantile Transformer
# Para distribuição uniforme
qt_uniform = QuantileTransformer(output_distribution='uniform', random_state=42)
X_qt_uniform = qt_uniform.fit_transform(X)
X_qt_uniform_df = pd.DataFrame(X_qt_uniform, columns=X.columns)

# Para distribuição normal
qt_normal = QuantileTransformer(output_distribution='normal', random_state=42)
X_qt_normal = qt_normal.fit_transform(X)
X_qt_normal_df = pd.DataFrame(X_qt_normal, columns=X.columns)

print("Estatísticas após Quantile Transform (Uniforme):")
print(X_qt_uniform_df.describe())

print("\nEstatísticas após Quantile Transform (Normal):")
print(X_qt_normal_df.describe())

In [ ]:
# @title
# Visualizando o efeito do Quantile Transformer
# Para uma variável com outliers
X_std = StandardScaler().fit_transform(X)
X_std_df = pd.DataFrame(X_std, columns=X.columns)

plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
sns.histplot(X[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Original')

plt.subplot(2, 2, 2)
sns.histplot(X_std_df[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Standard')

plt.subplot(2, 2, 3)
sns.histplot(X_qt_uniform_df[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Quantile (Uniform)')

plt.subplot(2, 2, 4)
sns.histplot(X_qt_normal_df[skewed_feature], kde=True)
plt.title(f'{skewed_feature} - Quantile (Normal)')

plt.tight_layout()
plt.show()

A Análise de Componentes Principais (PCA) é particularmente sensível à escala das variáveis, pois baseia-se na variância dos dados.

Quando as variáveis têm escalas muito diferentes, aquelas com maior variância podem dominar as primeiras componentes principais, mesmo que não contenham informações mais relevantes.

Vamos comparar os resultados do PCA com e sem escalonamento. (*Removendo do dataset os atributos Latitude e Longitude.*)

In [ ]:
# @title
# Removendo colunas Latitude e Longitude
Xa = X.drop(columns=["Latitude", "Longitude"])
# PCA sem escalonamento
pca_no_scaling = PCA(n_components=2)
X_pca_no_scaling = pca_no_scaling.fit_transform(Xa)

# PCA com StandardScaler
X_std = StandardScaler().fit_transform(Xa)
pca_with_scaling = PCA(n_components=2)
X_pca_with_scaling = pca_with_scaling.fit_transform(X_std)

# Criando DataFrames para visualização
pca_no_scaling_df = pd.DataFrame({
    'PC1': X_pca_no_scaling[:, 0],
    'PC2': X_pca_no_scaling[:, 1],
    'target': y
})

pca_with_scaling_df = pd.DataFrame({
    'PC1': X_pca_with_scaling[:, 0],
    'PC2': X_pca_with_scaling[:, 1],
    'target': y
})

In [ ]:
# @title
# Visualizando o resultado do PCA
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
scatter = plt.scatter(pca_no_scaling_df['PC1'], pca_no_scaling_df['PC2'],
                       c=pca_no_scaling_df['target'], cmap='viridis', alpha=0.5)
plt.colorbar(scatter, label='Preço')
plt.title('PCA sem Escalonamento')
plt.xlabel('Primeira Componente Principal')
plt.ylabel('Segunda Componente Principal')

plt.subplot(1, 2, 2)
scatter = plt.scatter(pca_with_scaling_df['PC1'], pca_with_scaling_df['PC2'],
                       c=pca_with_scaling_df['target'], cmap='viridis', alpha=0.5)
plt.colorbar(scatter, label='Preço')
plt.title('PCA com Standardization')
plt.xlabel('Primeira Componente Principal')
plt.ylabel('Segunda Componente Principal')

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Comparando a variância explicada
display(Markdown("## Variância explicada sem escalonamento:"))
print(f"PC1: {pca_no_scaling.explained_variance_ratio_[0]:.4f} ({pca_no_scaling.explained_variance_ratio_[0]*100:.2f}%)")
print(f"PC2: {pca_no_scaling.explained_variance_ratio_[1]:.4f} ({pca_no_scaling.explained_variance_ratio_[1]*100:.2f}%)")
print(f"Total: {sum(pca_no_scaling.explained_variance_ratio_[:2])*100:.2f}%")

display(Markdown("## Variância explicada com standardization:"))
print(f"PC1: {pca_with_scaling.explained_variance_ratio_[0]:.4f} ({pca_with_scaling.explained_variance_ratio_[0]*100:.2f}%)")
print(f"PC2: {pca_with_scaling.explained_variance_ratio_[1]:.4f} ({pca_with_scaling.explained_variance_ratio_[1]*100:.2f}%)")
print(f"Total: {sum(pca_with_scaling.explained_variance_ratio_[:2])*100:.2f}%")

In [ ]:
# @title
# Analisando os componentes de carregamento (loading components)
# Esse resultado mostra a contribuição de cada variável para as componentes principais

loadings_no_scaling = pd.DataFrame(
    pca_no_scaling.components_.T,
    columns=['PC1', 'PC2'],
    index=Xa.columns
)

loadings_with_scaling = pd.DataFrame(
    pca_with_scaling.components_.T,
    columns=['PC1', 'PC2'],
    index=Xa.columns
)

print("Carregamentos (loadings) sem escalonamento:")
print(loadings_no_scaling)

print("\nCarregamentos (loadings) com standardization:")
print(loadings_with_scaling)

In [ ]:
# @title
# Visualizando os carregamentos
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
sns.heatmap(loadings_no_scaling, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Carregamentos PCA sem Escalonamento')

plt.subplot(1, 2, 2)
sns.heatmap(loadings_with_scaling, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Carregamentos PCA com Standardization')

plt.tight_layout()
plt.show()

**Observações importantes sobre PCA e escalonamento:**

1. **Sem escalonamento**: Variáveis com maior variância dominam as componentes principais
2. **Com escalonamento**: Todas as variáveis contribuem de forma mais equilibrada
3. **Implicação prática**: Sempre escalone os dados antes de aplicar PCA, a menos que as diferenças de escala sejam significativas para a análise

# ==================================================
# 3. RECOMENDAÇÕES E MELHORES PRÁTICAS
# ==================================================

### QUANDO USAR CADA MÉTODO DE ESCALONAMENTO

1. **Min-Max Scaling (Normalização)**
   - Quando precisamos de valores em um intervalo específico, como [0, 1]
   - Quando a distribuição não é Gaussiana
   - Para análises que requerem valores em escalas comparáveis
   - Útil em visualizações e comparações diretas

2. **Standardization (Padronização)**
   - Para dados aproximadamente normais
   - Quando outliers não são significativos
   - Para análises estatísticas que assumem distribuição normal
   - Importante para PCA e outras técnicas de redução de dimensionalidade

3. **Robust Scaling**
   - Quando há muitos outliers nos dados
   - Para distribuições muito assimétricas
   - Quando a mediana e IQR são mais representativos que média e desvio padrão

4. **Power Transformer**
   - Para tornar dados mais "normais"
   - Quando a normalidade é um pressuposto importante
   - Para distribuições muito assimétricas
   - Útil antes de análises estatísticas paramétricas

5. **Quantile Transformer**
   - Para distribuições muito não-uniformes ou com outliers extremos
   - Quando a preservação de relações específicas não é crucial
   - Quando queremos garantir uma distribuição específica

### BOAS PRÁTICAS PARA ESCALONAMENTO

1. **Consistência no escalonamento**
   - Aplique o mesmo método de escalonamento em todas as análises relacionadas
   - Documente qual método foi usado e por quê
   - Mantenha os parâmetros do escalonamento para replicabilidade

2. **Trate diferentes tipos de variáveis adequadamente**
   - Não escalone variáveis categóricas (one-hot encoded)
   - Não escalone variáveis binárias (0/1)
   - Use transformadores específicos para diferentes tipos de variáveis

3. **Lide com outliers antes de escalonar, se necessário**
   - Considere remover ou tratar outliers antes de aplicar Min-Max Scaling
   - Para dados com muitos outliers, prefira Robust Scaling ou transformações

4. **Guarde os parâmetros do escalonamento**
   - Salve os parâmetros do scaler para uso futuro
   - Garanta que o mesmo escalonamento seja aplicado em novos dados
   - Documente os parâmetros utilizados

5. **Experimente diferentes métodos**
   - Compare como diferentes métodos afetam suas análises
   - Visualize as distribuições antes e depois do escalonamento
   - Escolha o método mais apropriado para seu objetivo específico

In [ ]:
# @title Exemplo de código para salvar e reutilizar um scaler
import joblib

# Criando e ajustando um scaler
scaler = StandardScaler()
scaler.fit(X)  # Ajustando nos dados

# Aplicando a transformação
X_scaled = scaler.transform(X)

# Salvando o scaler para uso futuro
joblib.dump(scaler, 'scaler.pkl')
print("Scaler salvo em 'scaler.pkl'")

# Carregando o scaler salvo
scaler_loaded = joblib.load('scaler.pkl')
print("Scaler carregado com sucesso!")

# Aplicando em novos dados
X_new_scaled = scaler_loaded.transform(X[:5])  # Exemplo com primeiras 5 linhas
print(f"\nDados transformados: {X_new_scaled.shape}")

---

## Resumo da Parte 2

Nesta parte avançada, aprendemos:

1. **Métodos avançados de escalonamento**:
   - MaxAbs Scaler, Normalizer, Power Transformer, Quantile Transformer
   - Cada um com casos de uso específicos

2. **Impacto em PCA**:
   - Escalonamento é crucial para PCA
   - Sem escalonamento, variáveis com maior variância dominam
   - Sempre escalone antes de aplicar PCA

3. **Melhores práticas**:
   - Quando usar cada método
   - Como implementar corretamente
   - Importância da consistência e documentação

**Próxima parte**: Exercício prático aplicando todas as técnicas aprendidas em um problema real.

---

## 🎯 Micro-atividade Guiada: Normalizando Distribuições Assimétricas (10 min)

**Objetivo**: Aplicar e comparar diferentes transformações para normalizar distribuições problemáticas.

**Cenário**: A variável 'AveOccup' (ocupação média por residência) está causando problemas em suas análises devido à sua extrema assimetria e presença de outliers. Vamos explorar sistematicamente diferentes transformações para resolver isso!

Trabalharemos passo a passo para encontrar a melhor solução! 🚀

### PASSO 1: Diagnóstico - Entendendo o Problema

Primeiro, vamos analisar a variável problemática para entender a natureza e gravidade da assimetria.
 - **Assimetria (Skewness):** Mede a simetria da distribuição em relação à média.

 - **Curtose (Kurtosis):** Mede o “peso das caudas” da distribuição comparado à normal.

In [ ]:
# @title
# PASSO 1: Analisando a variável problemática
variavel_problema = 'AveOccup'
dados_originais = X[variavel_problema].copy()

# Calculando estatísticas descritivas
print("📊 DIAGNÓSTICO DA VARIÁVEL 'AveOccup'")
print("=" * 50)
print(f"Média: {dados_originais.mean():.2f}")
print(f"Mediana: {dados_originais.median():.2f}")
print(f"Desvio padrão: {dados_originais.std():.2f}")
print(f"Mínimo: {dados_originais.min():.2f}")
print(f"Máximo: {dados_originais.max():.2f}")
print()
print(f"📈 Assimetria (Skewness): {scipy.stats.skew(dados_originais):.2f}")
print(f"📊 Curtose: {scipy.stats.kurtosis(dados_originais):.2f}")

# Teste de normalidade
statistic, p_value = scipy.stats.shapiro(dados_originais[:5000])  # Shapiro limita a 5000 amostras
print(f"\n🔬 Teste Shapiro-Wilk de normalidade:")
print(f"   p-valor: {p_value:.2e}")
if p_value < 0.05:
    print("   ❌ Rejeita normalidade (p < 0.05)")
else:
    print("   ✅ Não rejeita normalidade (p >= 0.05)")

# Visualização
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.hist(dados_originais, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(dados_originais.mean(), color='red', linestyle='--', label='Média')
plt.axvline(dados_originais.median(), color='green', linestyle='--', label='Mediana')
plt.title('Distribuição Original')
plt.xlabel(variavel_problema)
plt.ylabel('Frequência')
plt.legend()

plt.subplot(1, 3, 2)
plt.boxplot(dados_originais, vert=False)
plt.title('Boxplot - Detectando Outliers')
plt.xlabel(variavel_problema)

plt.subplot(1, 3, 3)
scipy.stats.probplot(dados_originais, dist="norm", plot=plt)
plt.title('Q-Q Plot - Comparação com Normal')

plt.suptitle('Análise Visual da Variável Problemática', fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 OBSERVAÇÕES:")
print("• Assimetria extremamente alta (>20) indica cauda muito longa à direita")
print("• Muitos outliers extremos visíveis no boxplot")
print("• Q-Q plot mostra forte desvio da normalidade")
print("• Necessária transformação para normalizar!")

### PASSO 2: Transformações Matemáticas Básicas

Vamos aplicar transformações matemáticas simples que frequentemente ajudam com assimetria positiva.

In [ ]:
# @title
# PASSO 2: Aplicando transformações matemáticas básicas

# Preparando dados (garantindo valores positivos para log e sqrt)
dados_positivos = dados_originais.copy()
min_val = dados_positivos.min()
if min_val <= 0:
    dados_positivos = dados_positivos - min_val + 0.001  # Tornando todos positivos

# Aplicando transformações
transformacoes = {
    'Original': dados_originais,
    'Log': np.log1p(dados_positivos),  # log(x+1) para evitar log(0)
    'Sqrt': np.sqrt(dados_positivos),
    'Reciprocal': 1 / (dados_positivos + 1)  # Evita divisão por zero
}

# Calculando assimetria para cada transformação
print("📊 ASSIMETRIA APÓS TRANSFORMAÇÕES BÁSICAS")
print("=" * 50)
for nome, dados_transf in transformacoes.items():
    skew = scipy.stats.skew(dados_transf)
    print(f"{nome:12} | Assimetria: {skew:8.2f} | ", end="")
    if abs(skew) < 0.5:
        print("✅ Excelente!")
    elif abs(skew) < 1:
        print("👍 Boa")
    elif abs(skew) < 2:
        print("⚠️  Moderada")
    else:
        print("❌ Alta")

# Visualizando todas as transformações
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (nome, dados_transf) in enumerate(transformacoes.items()):
    axes[i].hist(dados_transf, bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{nome}\nAssimetria: {scipy.stats.skew(dados_transf):.2f}')
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequência')

    # Adicionar linha vertical na média e mediana
    axes[i].axvline(dados_transf.mean(), color='red', linestyle='--', alpha=0.7, label='Média')
    axes[i].axvline(dados_transf.median(), color='green', linestyle='--', alpha=0.7, label='Mediana')
    axes[i].legend()

plt.suptitle('Comparação de Transformações Matemáticas Básicas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 ANÁLISE:")
print("• Log transformation reduz drasticamente a assimetria")
print("• Square root tem efeito moderado")
print("• Reciprocal inverte a direção da assimetria (útil para assimetria negativa)")

### PASSO 3: Power Transformers - Box-Cox e Yeo-Johnson

Agora vamos usar transformações mais sofisticadas que encontram automaticamente o melhor parâmetro λ.

In [ ]:
# @title
# PASSO 3: Aplicando Power Transformers

# Preparando dados para Box-Cox (precisa ser positivo)
dados_bc = dados_positivos.values.reshape(-1, 1)
dados_yj = dados_originais.values.reshape(-1, 1)

# Aplicando Box-Cox
pt_bc = PowerTransformer(method='box-cox', standardize=False)
dados_boxcox = pt_bc.fit_transform(dados_bc).flatten()

# Aplicando Yeo-Johnson
pt_yj = PowerTransformer(method='yeo-johnson', standardize=False)
dados_yeojohnson = pt_yj.fit_transform(dados_yj).flatten()

print("📊 POWER TRANSFORMERS - RESULTADOS")
print("=" * 50)
print(f"Box-Cox:")
print(f"  Lambda encontrado: {pt_bc.lambdas_[0]:.4f}")
print(f"  Assimetria após: {scipy.stats.skew(dados_boxcox):.2f}")

print(f"\nYeo-Johnson:")
print(f"  Lambda encontrado: {pt_yj.lambdas_[0]:.4f}")
print(f"  Assimetria após: {scipy.stats.skew(dados_yeojohnson):.2f}")

# Interpretando os lambdas
print("\n📚 INTERPRETAÇÃO DOS LAMBDAS:")
for nome, lambda_val in [("Box-Cox", pt_bc.lambdas_[0]), ("Yeo-Johnson", pt_yj.lambdas_[0])]:
    print(f"\n{nome} (λ = {lambda_val:.4f}):")
    if abs(lambda_val) < 0.1:
        print("  → Próximo de 0: transformação logarítmica")
    elif abs(lambda_val - 0.5) < 0.1:
        print("  → Próximo de 0.5: raiz quadrada")
    elif abs(lambda_val - 1) < 0.1:
        print("  → Próximo de 1: transformação linear (sem mudança)")
    elif lambda_val < 0:
        print("  → Negativo: transformação recíproca")
    else:
        print(f"  → Transformação de potência com λ = {lambda_val:.2f}")

# Visualizando
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(dados_originais, bins=50, edgecolor='black', alpha=0.7)
plt.title(f'Original\nAssimetria: {scipy.stats.skew(dados_originais):.2f}')
plt.xlabel('Valor')
plt.ylabel('Frequência')

plt.subplot(1, 3, 2)
plt.hist(dados_boxcox, bins=50, edgecolor='black', alpha=0.7, color='orange')
plt.title(f'Box-Cox (λ={pt_bc.lambdas_[0]:.3f})\nAssimetria: {scipy.stats.skew(dados_boxcox):.2f}')
plt.xlabel('Valor transformado')
plt.ylabel('Frequência')

plt.subplot(1, 3, 3)
plt.hist(dados_yeojohnson, bins=50, edgecolor='black', alpha=0.7, color='green')
plt.title(f'Yeo-Johnson (λ={pt_yj.lambdas_[0]:.3f})\nAssimetria: {scipy.stats.skew(dados_yeojohnson):.2f}')
plt.xlabel('Valor transformado')
plt.ylabel('Frequência')

plt.suptitle('Power Transformers - Comparação', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### PASSO 4: Quantile Transformer - Forçando Normalidade

O Quantile Transformer pode forçar qualquer distribuição a se tornar normal ou uniforme.

In [ ]:
# @title
# PASSO 4: Aplicando Quantile Transformer

# Preparando dados
dados_qt = dados_originais.values.reshape(-1, 1)

# Quantile Transformer - Distribuição Uniforme
qt_uniform = QuantileTransformer(output_distribution='uniform', random_state=42)
dados_qt_uniform = qt_uniform.fit_transform(dados_qt).flatten()

# Quantile Transformer - Distribuição Normal
qt_normal = QuantileTransformer(output_distribution='normal', random_state=42)
dados_qt_normal = qt_normal.fit_transform(dados_qt).flatten()

print("📊 QUANTILE TRANSFORMER - RESULTADOS")
print("=" * 50)
print(f"Distribuição Uniforme:")
print(f"  Assimetria: {scipy.stats.skew(dados_qt_uniform):.4f}")
print(f"  Min: {dados_qt_uniform.min():.4f}, Max: {dados_qt_uniform.max():.4f}")

print(f"\nDistribuição Normal:")
print(f"  Assimetria: {scipy.stats.skew(dados_qt_normal):.4f}")
print(f"  Média: {dados_qt_normal.mean():.4f}, Desvio: {dados_qt_normal.std():.4f}")

# Teste de normalidade para Quantile Normal
stat, p_val = scipy.stats.shapiro(dados_qt_normal[:5000])
print(f"\n🔬 Teste Shapiro-Wilk (Quantile Normal):")
print(f"  p-valor: {p_val:.4f}")
if p_val > 0.05:
    print("  ✅ Não rejeita normalidade!")
else:
    print("  ⚠️  Ainda rejeita normalidade (mas muito melhor)")

# Visualizando
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(dados_originais, bins=50, edgecolor='black', alpha=0.7)
plt.title(f'Original\nAssimetria: {scipy.stats.skew(dados_originais):.2f}')
plt.xlabel('Valor')
plt.ylabel('Frequência')

plt.subplot(1, 3, 2)
plt.hist(dados_qt_uniform, bins=50, edgecolor='black', alpha=0.7, color='purple')
plt.title(f'Quantile (Uniforme)\nAssimetria: {scipy.stats.skew(dados_qt_uniform):.4f}')
plt.xlabel('Valor transformado')
plt.ylabel('Frequência')

plt.subplot(1, 3, 3)
plt.hist(dados_qt_normal, bins=50, edgecolor='black', alpha=0.7, color='blue')
plt.title(f'Quantile (Normal)\nAssimetria: {scipy.stats.skew(dados_qt_normal):.4f}')
plt.xlabel('Valor transformado')
plt.ylabel('Frequência')

plt.suptitle('Quantile Transformer - Forçando Distribuições', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 OBSERVAÇÃO:")
print("• Quantile Transformer FORÇA a distribuição desejada")
print("• Extremamente eficaz, mas pode distorcer relações entre variáveis")
print("• Use com cuidado em análises que dependem de relações lineares")

### PASSO 5: Comparação Final e Recomendação

Vamos comparar todos os métodos e fazer uma recomendação baseada em evidências.

In [ ]:
# @title
# PASSO 5: Comparação final de todas as transformações (critério combinado)

import numpy as np
import pandas as pd
import scipy.stats
import matplotlib.pyplot as plt

# Compilando resultados
resultados = pd.DataFrame({
    'Transformação': ['Original', 'Log', 'Square Root', 'Box-Cox', 'Yeo-Johnson',
                      'Quantile Uniform', 'Quantile Normal'],
    'Assimetria': [
        scipy.stats.skew(dados_originais),
        scipy.stats.skew(np.log1p(dados_positivos)),
        scipy.stats.skew(np.sqrt(dados_positivos)),
        scipy.stats.skew(dados_boxcox),
        scipy.stats.skew(dados_yeojohnson),
        scipy.stats.skew(dados_qt_uniform),
        scipy.stats.skew(dados_qt_normal)
    ]
})

# Adicionar teste de normalidade (p-valor)
p_valores = []
for dados in [dados_originais, np.log1p(dados_positivos), np.sqrt(dados_positivos),
              dados_boxcox, dados_yeojohnson, dados_qt_uniform, dados_qt_normal]:
    stat, p = scipy.stats.shapiro(dados[:5000])
    p_valores.append(p)

resultados['P-valor (Shapiro)'] = p_valores

# Normalização para criar critério combinado
abs_skew = abs(resultados['Assimetria'])
skew_norm = (abs_skew - abs_skew.min()) / (abs_skew.max() - abs_skew.min())
pval_norm = (resultados['P-valor (Shapiro)'] - min(resultados['P-valor (Shapiro)'])) / \
            (max(resultados['P-valor (Shapiro)']) - min(resultados['P-valor (Shapiro)']))

# Score final: menor assimetria + maior p-valor
resultados['Score'] = (1 - skew_norm) + pval_norm

# Identificando o melhor método
melhor_metodo = resultados.sort_values('Score', ascending=False).iloc[0]['Transformação']

# Ordenar tabela para visualização
resultados = resultados.sort_values('Score', ascending=False)

print("🏆 TABELA COMPARATIVA FINAL COM CRITÉRIO COMBINADO")
print("=" * 80)
print(resultados[['Transformação', 'Assimetria', 'P-valor (Shapiro)', 'Score']].to_string(index=False))

print("\n🎯 RECOMENDAÇÃO (assimetria baixa + p-valor alto):", melhor_metodo)
print("=" * 80)

# Análise detalhada da recomendação
if melhor_metodo == 'Quantile Normal':
    print("\n📊 ANÁLISE DA RECOMENDAÇÃO:")
    print("• Quantile Transformer (Normal) produziu a distribuição mais próxima da normal")
    print("• Assimetria praticamente zero")
    print("• CUIDADO: O Quantile Normal transforma não-linearmente, pode distorcer relações")
    print("• USE quando: Normalidade é crucial e relações não-lineares são aceitáveis")
    print("• EVITE quando: Interpretabilidade dos coeficientes é importante")
elif melhor_metodo in ['Box-Cox', 'Yeo-Johnson']:
    print("\n📊 ANÁLISE DA RECOMENDAÇÃO:")
    print(f"• {melhor_metodo} oferece bom equilíbrio entre normalização e preservação de relações")
    print("• Transformação paramétrica com λ otimizado")
    print("• Preserva relações monotônicas")
    print("• BOA ESCOLHA para a maioria dos casos práticos")
elif melhor_metodo == 'Log':
    print("\n📊 ANÁLISE DA RECOMENDAÇÃO:")
    print("• Transformação logarítmica é simples e interpretável")
    print("• Excelente para dados com crescimento exponencial")
    print("• Preserva ordem e é reversível")
    print("• IDEAL quando a interpretabilidade é importante")
else:
    print("\n📊 Análise adicional: método selecionado pelo critério combinado.")

# Visualização final comparativa
plt.figure(figsize=(16, 8))

transformacoes_finais = {
    'Original': dados_originais,
    'Log': np.log1p(dados_positivos),
    'Box-Cox': dados_boxcox,
    'Yeo-Johnson': dados_yeojohnson,
    'Quantile Normal': dados_qt_normal
}

for i, (nome, dados) in enumerate(transformacoes_finais.items(), 1):
    plt.subplot(2, 3, i)
    plt.hist(dados, bins=40, edgecolor='black', alpha=0.7)
    plt.title(f'{nome}\nSkew: {scipy.stats.skew(dados):.3f}')
    plt.xlabel('Valor')
    plt.ylabel('Frequência')

    # Destacar o melhor método
    if nome == melhor_metodo:
        plt.gca().patch.set_facecolor('lightgreen')
        plt.gca().patch.set_alpha(0.3)

plt.suptitle('Comparação Visual Final - Melhor Método Destacado em Verde',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🎓 LIÇÕES APRENDIDAS:")
print("1. Dados com assimetria extrema (>20) precisam de transformação forte")
print("2. Power Transformers encontram automaticamente a melhor transformação")
print("3. Quantile Transformer garante normalidade mas pode distorcer relações")
print("4. Sempre teste múltiplos métodos e compare resultados")
print("5. Considere o trade-off entre normalidade e interpretabilidade")
